# BorakBot — Base-model bake-off (Kaggle)

Picks the QLoRA base model by measuring the candidates instead of asserting one.
Produces `docs/model_selection.md`'s evidence: a blinded Likert sheet and an
objective refusal table.

**Settings → Accelerator → GPU T4 x2** before running. On CPU this will not finish.

Run the cells **in order**. Everything outside `/kaggle/working` vanishes when the
session ends, and a session is capped at 12 hours.

    code     -> GitHub
    models   -> Hugging Face (cached in /kaggle/working after first pull)
    results  -> /kaggle/working, downloaded at the end

This notebook does NOT train anything. It only generates replies and counts
refusals; the human rating happens off-Kaggle in a spreadsheet.


## Cell 1 — Setup

Installs the generation stack and clones the repo. Two to three minutes.
Re-run after every session restart.

`peft` is installed here even though nothing is fine-tuned yet, because the same
`generate.py` is re-run at Step 4 with `--adapter`.


In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate

!rm -rf /kaggle/working/NLP-BorakBot
!git clone -q https://github.com/yongvay/NLP-BorakBot.git /kaggle/working/NLP-BorakBot

%cd /kaggle/working/NLP-BorakBot
!git log --oneline -1

import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## Cell 2 — Hugging Face token

**Required.** `meta-llama/Llama-3.2-3B-Instruct` is gated, and it is one of the two
candidates. Access has been granted on the account, but the notebook still has to
authenticate as that account to pull the weights.

Add the token under **Add-ons → Secrets** as `HF_TOKEN`. Never paste it into a cell —
notebook source is committed, output is shared, and a pasted token is a leaked token.

MaLLaM is ungated and will still load if this cell fails, so a failure here shows up
as one candidate missing rather than as an error. Check the printed confirmation.


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
    print('HF login ok')
except Exception as e:
    print('no HF_TOKEN secret — ungated models will still work')
    print(' ', e)


## Cell 3 — Generate

Each candidate answers the same 20 committed probes from `eval/probe_set.jsonl`.
Roughly 5–10 minutes per model on a T4, plus a one-off ~6 GB download each.

The two candidates come from different families on purpose: Llama-3.2-3B-Instruct is
the strong general instruct model, `mallam-3b-20k-instructions` is MaLLaM, trained on
Malaysian instructions. That is the contrast worth measuring for a rojak task.

**MaLLaM uses the Mistral `[INST]` chat template, not llama3.** If it wins,
`training/qlora_config.yaml` needs `template: mistral`. A wrong template does not
raise — it trains on mis-delimited text and yields a model that rambles past its
stop token.

4-bit loading is used here for the same reason it is used in training: it is the
configuration the model will actually be judged in. Comparing candidates at fp16 and
then deploying at 4-bit would measure a setup nobody ships.


In [ ]:
CANDIDATES = {
    'llama':  'meta-llama/Llama-3.2-3B-Instruct',       # gated, access granted
    'mallam': 'mesolitica/mallam-3b-20k-instructions',  # MaLLaM, Mistral-based
    # 'qwen': 'Qwen/Qwen2.5-3B-Instruct',               # ungated; add for a third
}

for tag, model_id in CANDIDATES.items():
    print()
    print('=' * 70)
    print(f'{tag}: {model_id}')
    print('=' * 70, flush=True)
    !python eval/generate.py --model {model_id} --tag {tag} --4bit


## Cell 4 — Objective refusal table

Fallback accuracy and over-refusal, exact and loose. Instant, no GPU.

Expect both to score badly here — neither has been taught the fallback line, so
`exact` will likely be 0% everywhere. That is not a broken script; it is the
measurement that makes the fine-tuned comparison at Step 4 mean something. What to
read at this stage is `loose`: whether a candidate declines at all when it should.


In [ ]:
!python eval/refusal_report.py --runs {','.join(CANDIDATES)} --show-misses


## Cell 5 — Build the rating sheet and save everything

**Do not skip.** Results written inside the repo clone are lost when the session ends;
`/kaggle/working` is what persists and what the Output panel offers for download.

`rating_key.json` un-blinds the sheet — leave it unopened until both members have
finished rating.


In [ ]:
!python eval/make_rating_sheet.py --runs {','.join(CANDIDATES)}

import shutil, pathlib
OUT = pathlib.Path('/kaggle/working/bakeoff'); OUT.mkdir(parents=True, exist_ok=True)
SRC = pathlib.Path('/kaggle/working/NLP-BorakBot/eval')

for name in ['rating_sheet.csv', 'rating_key.json', 'probe_set.jsonl']:
    shutil.copy2(SRC / name, OUT / name)
shutil.copytree(SRC / 'results', OUT / 'results', dirs_exist_ok=True)

for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'{p.stat().st_size:>9,}  {p.relative_to(OUT)}')


## Cell 6 — What happens next

1. Download `bakeoff/` from the **Output** panel on the right.
2. Copy `eval/results/*.json` and `eval/rating_sheet.csv` into the repo and commit them.
   The generations are the evidence behind the choice; a table without them is an
   assertion.
3. Each member takes their own copy of the sheet, rates every row 1–5 on the three
   axes, and saves it as `eval/rating_sheet_<initials>.csv`. Rate independently —
   comparing as you go destroys the agreement number.
4. Locally: `python eval/score_ratings.py`
5. Write `docs/model_selection.md`: the table, the winner, one paragraph of why.
   If the spread is under ~0.3 the candidates are tied — pick on tooling risk and
   say so plainly.

Then Step 3: convert the splits to LLaMA-Factory format and train.
